# Week 3 · Try It: Predict Remaining Flight Time with Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weisongwen/AAE4011-S22627/blob/main/Lecture%20Notes/Week%203/linreg_flight_time.ipynb)

AAE4011 Artificial Intelligence in Unmanned Autonomous Systems · Semester 2, 2026/27

40 logged flights of a drone with a 4S LiPo battery: the battery voltage `V` (volts) and the flight time that remained `t` (minutes). The data are simulated around the slide *UAS Case: Estimating Remaining Flight Time*.

**Tasks**
1. Fit `t = aV + b` with the least-squares formulas from the slides, and compute R² (Part 1).
2. Fit the same line with gradient descent and try four learning rates (Part 2).
3. The drone needs 8 minutes to fly home and land. At what voltage must it turn back (Part 3)?

## Part 1 · Least squares in closed form

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(4011)
V = np.sort(rng.uniform(14.2, 16.6, 40))            # battery voltage (V)
t = 8.9 * V - 125.2 + rng.normal(0, 1.2, 40)        # remaining flight time (min)

Vbar, tbar = V.mean(), t.mean()
Sxy = np.sum((V - Vbar) * (t - tbar))
Sxx = np.sum((V - Vbar) ** 2)
a = Sxy / Sxx
b = tbar - a * Vbar
pred = a * V + b
R2 = 1 - np.sum((t - pred) ** 2) / np.sum((t - tbar) ** 2)
print(f"a = {a:.2f} min/V,  b = {b:.1f} min,  R^2 = {R2:.3f}")

plt.scatter(V, t, s=18, label="flights")
plt.plot(V, pred, "C1", label=f"t = {a:.2f} V {b:+.1f}")
plt.xlabel("battery voltage V (V)"); plt.ylabel("remaining flight time t (min)"); plt.legend()
plt.show()

**Question.** Check your answer with `np.polyfit(V, t, 1)`. Do the two agree?

## Part 2 · Gradient descent and the learning rate

We centre the input, `x = V − mean(V)`, so that both parameters learn at a similar speed (try it without centring afterwards and see what happens).

In [ ]:
x = V - Vbar

def gd(alpha, steps=50):
    a, b = 0.0, 0.0
    J = []
    for _ in range(steps):
        e = a * x + b - t                  # errors
        J.append(np.mean(e ** 2) / 2)
        a -= alpha * np.mean(e * x)        # dJ/da
        b -= alpha * np.mean(e)            # dJ/db
    return a, b, J

for alpha in (0.01, 0.1, 1.0, 2.05):     # change these and run again
    a_gd, b_gd, J = gd(alpha)
    print(f"alpha = {alpha:<5}  a = {a_gd:9.3f}  J after 50 steps = {J[-1]:.4g}")
    plt.semilogy(J, label=f"alpha = {alpha}")
plt.xlabel("iteration"); plt.ylabel("loss J (log scale)"); plt.legend(); plt.show()

**Questions.** Which learning rate is too small, which is about right, and which diverges? (Hint: for centred data the loss is a bowl whose steepest direction has curvature 1, so any `alpha` above 2 diverges.)

## Part 3 · A return-to-home rule

In [ ]:
need = 8.0                                 # minutes to fly home and land
V_rth = (need - b) / a
print(f"turn back when V drops below {V_rth:.2f} V")

**Question.** Why would you add a safety margin on top of this voltage? Give two reasons from the slide *Limitations → better models*.